# Test 1: burbujas con promedios de cinco semillas

Cada burbuja resume las semillas 42, 99, 123, 777 y 2024: X = media de FPR (%), Y = media de F1, área = media del número de fuentes con al menos un flujo verdadero positivo. Se promedian las métricas individuales, cada una con su propio umbral seleccionado en validación; no se combinan las predicciones ni se calcula una unión de fuentes.

Incluye MLP, E-GraphSAGE y Static GNN reentrenados; las cinco corridas históricas de EdgeGRU_NoX_BiasOn; y los factoriales M/P, incluidos los ajustados por parámetros por defecto. M1_P1 representa ST-GNN con la política corregida de memoria. No incluye ST-GNN histórico.

**Uso:** ejecutar la primera celda en el Colab ya configurado y con Drive montado; no hace falta reiniciar el runtime. Lee los grafos de Test 1 una vez y realiza cinco pasadas de inferencia para recuperar las fuentes de Edge GRU histórico; no entrena modelos. Verifica los conteos TP/FP/TN/FN contra los históricos y se detiene si no coinciden. Usa los umbrales del NPZ histórico si están disponibles; si no, los del CSV. Esta verificación evita asumir igualdad por un redondeo del umbral.

Luego ejecutar la segunda celda. Para mostrar dispersión, cambiar `SHOW_STD = True` y repetir únicamente esa segunda celda. Las barras muestran ±1 desviación estándar muestral entre semillas (no intervalos de confianza); se recortan en los límites válidos de cada métrica. La tabla conserva las desviaciones completas, incluida la de fuentes; la burbuja siempre representa el promedio de fuentes.

Se guardan PNG, PDF y CSV con promedios y datos individuales en una carpeta nueva bajo `results_comparison/primary_comparison_figures/`. El cálculo real requiere los archivos de Drive; el gráfico fue probado localmente con datos de ejemplo.


In [ ]:
# Usar el mismo runtime de Colab, con Drive montado y el repositorio disponible.
import sys
from pathlib import Path
for candidate in (Path('/content/temporalgnn-nids/code/python'), Path.cwd() / 'code/python'):
    if (candidate / 'utils/models.py').is_file():
        sys.path.insert(0, str(candidate))
        break

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import torch
from utils.datasets import NF_IDS_Dataset
from utils.models import EdgeGRU_Baseline_NoX

ROOT = Path('/content/drive/MyDrive/nids-mitre')
BASE = ROOT / 'results_baseline_retraining_factorial_protocol_v1'
FACT = ROOT / 'results_stgnn_memory_message_factorial_v1'
HIST = ROOT / 'results_earlystopping'
INCLUDE_PARAMETER_MATCHED = True
SEEDS = {42, 123, 777, 2024, 99}
torch.set_num_threads(1)

baselines = pd.read_csv(BASE / 'test1_all_seeds.csv', float_precision='round_trip')
baselines = baselines.loc[baselines['model'].isin(['MLP', 'E-GraphSAGE', 'Static GNN'])].copy()
assert set(baselines['model']) == {'MLP', 'E-GraphSAGE', 'Static GNN'}
assert baselines['status'].eq('complete').all()
baselines['origin'] = 'retrained baseline'
factorial = pd.read_csv(FACT / 'test1_all_seeds.csv', float_precision='round_trip')
factorial = factorial.loc[factorial['model'].str.match(r'^M[01]_P[01](?:_param)?$')].copy()
if not INCLUDE_PARAMETER_MATCHED:
    factorial = factorial.loc[~factorial['model'].str.endswith('_param')].copy()
assert {'M0_P0', 'M0_P1', 'M1_P0', 'M1_P1'} <= set(factorial['model'])
assert factorial['status'].eq('complete').all()
factorial['origin'] = 'factorial retraining'

# Recover source counts for ALL FIVE historical Edge GRU NoX checkpoints.
experiment = 'EdgeGRU_NoX_BiasOn'
historical = pd.read_csv(HIST / 'test1_results' / f'test1_metrics_{experiment}.csv',
                         float_precision='round_trip')
assert len(historical) == 5 and set(historical['seed'].astype(int)) == SEEDS
assert historical['seed'].is_unique
manifest = json.loads((FACT / 'manifest.json').read_text())
split = manifest['splits']['test1']
dataset = NF_IDS_Dataset(root_dir=split['root'], split=split['directory'])
assert len(dataset) == split['graphs']
print('Loading Test 1 graphs once; no training is performed.', flush=True)
graphs = []
for i in range(len(dataset)):
    graph = dataset[i].cpu()
    if graph.x.shape[0] > 0:
        graphs.append(graph)
    if (i + 1) % 100 == 0:
        print(f'Read {i + 1}/{len(dataset)} graphs', flush=True)
y = np.concatenate([g.y.numpy().reshape(-1) for g in graphs]).astype(np.int8)
sources = np.concatenate([g.global_node_ids[g.edge_index[0]].numpy() for g in graphs])
assert len(y) == 191679 and int(y.sum()) == 3434
assert np.unique(sources[y == 1]).size == 16

threshold_file = HIST / 'logs' / experiment / f'thresholds_{experiment}.npz'
thresholds = {}
if threshold_file.exists():
    with np.load(threshold_file, allow_pickle=False) as saved:
        thresholds = {key: float(saved[key]) for key in saved.files}

edge_rows = []
for _, old in historical.iterrows():
    seed = int(old['seed'])
    threshold = thresholds.get(f'seed_{seed}', float(old['optimal_threshold']))
    paths = list((HIST / 'saved_models' / experiment).glob(f'{old["run_id"]}_*.pth'))
    assert len(paths) == 1, (seed, paths)
    model = EdgeGRU_Baseline_NoX(**manifest['model_params']).cpu()
    model.load_state_dict(torch.load(paths[0], map_location='cpu', weights_only=True), strict=True)
    model.eval()
    model.reset_memory()
    print(f'Evaluating historical Edge GRU seed {seed}...', flush=True)
    with torch.no_grad():
        parts = [torch.sigmoid(model(g.x, g.edge_index, g.edge_attr,
                 g.global_node_ids).view(-1)).numpy() for g in graphs]
    probabilities = np.concatenate(parts)
    assert np.isfinite(probabilities).all() and len(probabilities) == len(y)
    alert = probabilities > threshold
    counts = dict(TP=int(((y == 1) & alert).sum()), FP=int(((y == 0) & alert).sum()),
                  TN=int(((y == 0) & ~alert).sum()), FN=int(((y == 1) & ~alert).sum()))
    assert all(counts[key] == int(old[key]) for key in counts), (
        f'Historical seed {seed}: inference does not match recorded counts. '
        f'Check its threshold/environment. Actual={counts}')
    detected = int(np.unique(sources[(y == 1) & alert]).size)
    edge_rows.append(dict(model='Edge GRU (historical NoX)', seed=seed,
        origin='historical baseline', F1=float(old['F1']), FPR=float(old['FPR']),
        sources_detected=detected, attacked_sources=16, flows=len(y), attacks=int(y.sum()),
        threshold=threshold, run_id=old['run_id'],
        checkpoint_sha256=hashlib.sha256(paths[0].read_bytes()).hexdigest()))
    print(f'  Verified: F1={old["F1"]:.6f}; detected sources={detected}', flush=True)
    model.reset_memory()
    del model, parts, probabilities

mean_seed_data = pd.concat([factorial, baselines, pd.DataFrame(edge_rows)], ignore_index=True)
assert not mean_seed_data.duplicated(['model', 'seed']).any()
assert mean_seed_data['flows'].eq(191679).all()
assert mean_seed_data['attacks'].eq(3434).all()
assert mean_seed_data['attacked_sources'].eq(16).all()
for name, group in mean_seed_data.groupby('model'):
    assert len(group) == 5 and set(group['seed'].astype(int)) == SEEDS, name
assert np.isfinite(mean_seed_data[['F1', 'FPR', 'sources_detected']].to_numpy()).all()
mean_seed_data['FPR_percent'] = 100 * mean_seed_data['FPR']
mean_summary = mean_seed_data.groupby('model', sort=False).agg(
    F1_mean=('F1', 'mean'), F1_std=('F1', 'std'),
    FPR_mean_percent=('FPR_percent', 'mean'), FPR_std_percent=('FPR_percent', 'std'),
    sources_mean=('sources_detected', 'mean'), sources_std=('sources_detected', 'std'),
    n_seeds=('seed', 'nunique'))
order = ['M0_P0', 'M0_P1', 'M1_P0', 'M1_P1', 'M0_P0_param', 'M1_P0_param',
         'MLP', 'E-GraphSAGE', 'Edge GRU (historical NoX)', 'Static GNN']
mean_summary = mean_summary.loc[[name for name in order if name in mean_summary.index]].reset_index()
mean_summary['plot_id'] = np.arange(1, len(mean_summary) + 1)
print('Ready: each row summarizes five seeds. Run the plotting cell next.', flush=True)


In [ ]:
from datetime import datetime, timezone
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

SHOW_STD = False  # True: horizontal and vertical bars for +/- one sample SD.
AREA_PER_SOURCE = 80.0
summary = mean_summary.copy()
palette = plt.get_cmap('tab20')
colors = {name: palette(i) for i, name in enumerate(summary['model'])}
fig, ax = plt.subplots(figsize=(15, 8))
for _, row in summary.sort_values('sources_mean', ascending=False).iterrows():
    x, y = row['FPR_mean_percent'], row['F1_mean']
    if SHOW_STD:
        # Truncate displayed bars at metric bounds; full SDs remain in the table.
        xerr = [[min(x, row['FPR_std_percent'])],
                [min(100 - x, row['FPR_std_percent'])]]
        yerr = [[min(y, row['F1_std'])], [min(1 - y, row['F1_std'])]]
        ax.errorbar(x, y, xerr=xerr, yerr=yerr, fmt='none', capsize=3,
                    ecolor=colors[row['model']], alpha=0.5, linewidth=1.2, zorder=2)
    ax.scatter(x, y, s=AREA_PER_SOURCE * row['sources_mean'],
        color=colors[row['model']], alpha=0.7, edgecolor='black', linewidth=0.9, zorder=3)
    ax.annotate(str(int(row['plot_id'])), (x, y), ha='center', va='center',
                fontsize=9, fontweight='bold', zorder=4)

handles = []
for _, row in summary.iterrows():
    name = 'M1_P1 (ST-GNN, corrected memory)' if row['model'] == 'M1_P1' else row['model']
    label = (f'{int(row["plot_id"]):2d}. {name}\n'
             f'     5 seeds | mean sources: {row["sources_mean"]:.1f}')
    handles.append(Line2D([], [], linestyle='', marker='o', markersize=8,
        color=colors[row['model']], markeredgecolor='black', label=label))
legend = ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.02, 1),
                   frameon=False, fontsize=9, labelspacing=0.85)
ax.add_artist(legend)
size_handles = [ax.scatter([], [], s=AREA_PER_SOURCE * count, color='lightgray',
    edgecolor='black', alpha=0.7, label=f'{count} sources') for count in (4, 8, 12)]
ax.legend(handles=size_handles, title='Mean sources (circle area)', loc='lower right',
          labelspacing=2.6, borderpad=1.6)
right = summary['FPR_mean_percent'] + (summary['FPR_std_percent'] if SHOW_STD else 0)
ax.set_xlim(0, max(1.0, float(right.max()) * 1.15))
ax.set_ylim(-0.035, 1.035)
ax.set_xlabel('Mean false-positive rate (%) — lower is better')
ax.set_ylabel('Mean F1 — higher is better')
ax.set_title('Test 1: five-seed means — factorial models and baselines')
ax.grid(alpha=0.22)
ax.set_axisbelow(True)
fig.subplots_adjust(right=0.65, bottom=0.17)
note = ('Bars: +/- 1 sample SD, truncated at metric bounds; not confidence intervals.'
        if SHOW_STD else 'Between-seed SD is reported in the table; bars are hidden.')
fig.text(0.1, 0.04,
    'Arithmetic means of five per-seed metrics, each at its own validation-selected threshold.\n'
    'Circle area = mean number of sources with at least one TP per seed (16 attacked sources).\n' + note,
    ha='left', fontsize=9)
out = (ROOT / 'results_comparison' / 'primary_comparison_figures' /
       datetime.now(timezone.utc).strftime('test1_means_%Y%m%dT%H%M%S_%fZ'))
out.mkdir(parents=True, exist_ok=False)
for extension in ('png', 'pdf'):
    fig.savefig(out / f'test1_mean_bubbles.{extension}', dpi=220, bbox_inches='tight')
summary.to_csv(out / 'test1_mean_summary.csv', index=False)
mean_seed_data.to_csv(out / 'test1_individual_seed_data.csv', index=False)
plt.show()
display(summary.round(6))
print('Saved:', out)
